# LeWM — Push-T Sanity Run

Runs LeWorldModel (Maes et al. 2026, arXiv 2603.19312) on Push-T using the **official le-wm repo**.
Goal is fast sanity validation and artifact generation (not full paper replication).

**Target:** verify training/eval pipeline works and produce a usable checkpoint for downstream tests
**Typical runtime:** ~1 h on A100 40 GB
**Disk needed:** ~25 GB for dataset + checkpoints/logs

**Prerequisites:**
- Runtime -> Change runtime type -> **A100**
- Colab Pro / Pro+ for A100 access

---
## Steps
1. GPU check (A100-only settings)
2. Install dependencies
3. Download Push-T dataset from HuggingFace
4. Clone le-wm
5. Train with step-based checkpointing
6. Run evaluation (CEM-MPC on PushT-v1)
7. [Optional] Upload checkpoint to S3


In [ ]:
# ── 1. GPU check + A100 fixed hyperparams ───────────────────────────────────
import subprocess, shutil, torch

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU found. Runtime → Change runtime type → A100')
print('GPU:', result.stdout.strip())

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'VRAM: {vram_gb:.0f} GB')
if vram_gb < 35:
    raise RuntimeError('This notebook is configured for A100 runs only (>=35 GB VRAM).')

# A100-only defaults
BATCH_SIZE  = 512
NUM_WORKERS = 8
PRECISION   = 'bf16-mixed'

print(f'→ A100 mode: batch_size={BATCH_SIZE}  num_workers={NUM_WORKERS}  precision={PRECISION}')

total, used, free = shutil.disk_usage('/')
print(f'Disk: {free / 1e9:.1f} GB free / {total / 1e9:.1f} GB total')
if free / 1e9 < 25:
    print('WARNING: less than 25 GB free. Dataset may not fit — use a high-RAM runtime.')


In [ ]:
# ── 2. Install dependencies ────────────────────────────────────────────────
# box2d-py (pulled in by stable-worldmodel[env]) builds from source and
# needs swig + build tools. Install those first.

!apt-get install -y -q swig build-essential
!pip install -q 'stable-worldmodel[train,env]' einops 'numpy<2.0.0' zstandard huggingface_hub
print('Install complete')

In [ ]:
# ── 3. Configure paths ─────────────────────────────────────────────────────
import os
from pathlib import Path

STABLEWM_HOME = Path('/content/stable-wm')
STABLEWM_HOME.mkdir(parents=True, exist_ok=True)
os.environ['STABLEWM_HOME'] = str(STABLEWM_HOME)

# [Optional] Weights & Biases — set to False to skip W&B logging
USE_WANDB = False
if USE_WANDB:
    import wandb
    wandb.login()  # prompts for API key

print(f'STABLEWM_HOME = {STABLEWM_HOME}')
print(f'W&B logging: {USE_WANDB}')

In [ ]:
# ── 4. Download Push-T dataset from HuggingFace ────────────────────────────
# Dataset: quentinll/lewm-pusht on HuggingFace (public, no auth needed)
# File: pusht_expert_train.h5.zst (~13 GB compressed, ~18-22 GB decompressed)
# Download takes ~5-10 min on Colab.

import os, zstandard, urllib.request
from pathlib import Path
from huggingface_hub import hf_hub_download

H5_PATH = STABLEWM_HOME / 'pusht_expert_train.h5'
ZST_PATH = STABLEWM_HOME / 'pusht_expert_train.h5.zst'

if H5_PATH.exists():
    print(f'Dataset already present: {H5_PATH} ({H5_PATH.stat().st_size / 1e9:.1f} GB)')
else:
    if not ZST_PATH.exists():
        print('Downloading pusht_expert_train.h5.zst from HuggingFace (~13 GB) ...')
        downloaded = hf_hub_download(
            repo_id='quentinll/lewm-pusht',
            filename='pusht_expert_train.h5.zst',
            repo_type='dataset',
            local_dir=str(STABLEWM_HOME),
        )
        print(f'Downloaded to: {downloaded}')
        # hf_hub_download may return a cached path; ensure it's where we expect
        zst_actual = Path(downloaded)
        if zst_actual != ZST_PATH:
            import shutil
            shutil.copy2(zst_actual, ZST_PATH)
    else:
        print(f'Compressed file already present: {ZST_PATH}')

    print(f'Decompressing to {H5_PATH} ...')
    dctx = zstandard.ZstdDecompressor()
    with open(ZST_PATH, 'rb') as ifh, open(H5_PATH, 'wb') as ofh:
        dctx.copy_stream(ifh, ofh)
    print(f'Done. H5 size: {H5_PATH.stat().st_size / 1e9:.1f} GB')
    ZST_PATH.unlink()  # free disk space

# Quick sanity check
import h5py
with h5py.File(H5_PATH, 'r') as f:
    keys = list(f.keys())
    n_rows = len(f[keys[0]])
print(f'Dataset keys: {keys}')
print(f'Total rows:   {n_rows:,}')

In [ ]:
# ── 5. Clone le-wm ─────────────────────────────────────────────────────────
import os
from pathlib import Path

LEWM_DIR = Path('/content/le-wm')
if LEWM_DIR.exists():
    print(f'le-wm already present at {LEWM_DIR}')
else:
    !git clone --depth 1 https://github.com/lucas-maes/le-wm.git /content/le-wm

os.chdir(LEWM_DIR)
print(f'Working directory: {os.getcwd()}')
!ls config/train/data/

In [ ]:
# -- 6. Train with step-based checkpointing -----------------------------------
# A100 40 GB defaults: batch=512, workers=8, bf16-mixed
# Saves model-object checkpoints every CHECKPOINT_EVERY_N_STEPS so short runs
# still produce artifacts even if epoch 0 does not finish.

import os, subprocess
from pathlib import Path
os.environ['STABLEWM_HOME'] = str(STABLEWM_HOME)

CHECKPOINT_EVERY_N_STEPS = 500
MAX_EPOCHS = 100

# Hotfix: bf16 validation can fail if batch['action'] stays float32.
# Patch le-wm/train.py to cast action to action_encoder dtype before encode().
train_py = Path('/content/le-wm/train.py')
text = train_py.read_text()
old = "output = self.model.encode(batch)"
broken = "if 'action' in batch:\\n        act_dtype = self.model.action_encoder.patch_embed.weight.dtype\\n        batch['action'] = batch['action'].to(dtype=act_dtype)\\n    output = self.model.encode(batch)"
fixed = '''if 'action' in batch:
        act_dtype = self.model.action_encoder.patch_embed.weight.dtype
        batch['action'] = batch['action'].to(dtype=act_dtype)
    output = self.model.encode(batch)'''

if broken in text:
    text = text.replace(broken, fixed)
    train_py.write_text(text)
    print('Repaired escaped-newline hotfix in /content/le-wm/train.py')
elif old in text and "act_dtype = self.model.action_encoder.patch_embed.weight.dtype" not in text:
    train_py.write_text(text.replace(old, fixed, 1))
    print('Applied bf16 action dtype hotfix in /content/le-wm/train.py')
else:
    print('bf16 hotfix already present')

# Patch utils.py: add step-based save hook to ModelObjectCallBack.
utils_py = Path('/content/le-wm/utils.py')
u_text = utils_py.read_text()
if 'step_interval' not in u_text:
    u_text = u_text.replace(
        'def __init__(self, dirpath, filename="model_object", epoch_interval: int = 1):',
        'def __init__(self, dirpath, filename="model_object", epoch_interval: int = 1, step_interval: int = 0):'
    )
    u_text = u_text.replace('        self.epoch_interval = epoch_interval',
                            '        self.epoch_interval = epoch_interval\n        self.step_interval = int(step_interval)')
    marker = '    def _dump_model(self, model, path):\n'
    hook = '''    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        super().on_train_batch_end(trainer, pl_module, outputs, batch, batch_idx)

        if self.step_interval <= 0:
            return
        if not trainer.is_global_zero:
            return

        global_step = int(trainer.global_step)
        if global_step <= 0 or (global_step % self.step_interval) != 0:
            return

        output_path = self.dirpath / f"{self.filename}_step_{global_step}_object.ckpt"
        self._dump_model(pl_module.model, output_path)

'''
    if marker in u_text:
        u_text = u_text.replace(marker, hook + marker)
        utils_py.write_text(u_text)
        print('Patched utils.py for step-based object checkpointing')
    else:
        print('WARNING: Could not find insertion marker in utils.py')
else:
    print('utils.py step checkpoint hook already present')

# Patch train.py: pass step_interval into callback.
text = train_py.read_text()
call_old = 'object_dump_callback = ModelObjectCallBack(\n        dirpath=run_dir, filename=cfg.output_model_name, epoch_interval=1,\n    )'
call_new = (
    'object_dump_callback = ModelObjectCallBack(\n'
    '        dirpath=run_dir,\n'
    '        filename=cfg.output_model_name,\n'
    '        epoch_interval=1,\n'
    '        step_interval=cfg.get("checkpoint_every_n_steps", 0),\n'
    '    )'
)
if 'checkpoint_every_n_steps' not in text and call_old in text:
    train_py.write_text(text.replace(call_old, call_new, 1))
    print('Patched train.py callback wiring for step-based checkpoints')
else:
    print('train.py callback wiring already present (or pattern not found)')

wandb_flag = '' if USE_WANDB else 'wandb.enabled=false'
train_cmd = (
    f"set -o pipefail; HYDRA_FULL_ERROR=1 python3 train.py "
    f"data=pusht subdir=pusht {wandb_flag} "
    f"trainer.max_epochs={MAX_EPOCHS} loader.batch_size={BATCH_SIZE} "
    f"loader.num_workers={NUM_WORKERS} loader.pin_memory=false loader.persistent_workers=false "
    f"trainer.precision={PRECISION} +checkpoint_every_n_steps={CHECKPOINT_EVERY_N_STEPS} "
    f"2>&1 | tee /content/train_log.txt"
)
ret = subprocess.run(['bash', '-lc', train_cmd])
if ret.returncode != 0:
    raise RuntimeError(f"Training failed with exit code {ret.returncode}. Check /content/train_log.txt")

print('Training finished successfully (see /content/train_log.txt)')


In [ ]:
# -- 7. Locate and link checkpoint -------------------------------------------
import glob, shutil
from pathlib import Path

run_dir = STABLEWM_HOME / 'pusht'
epoch_ckpts = sorted(glob.glob(str(run_dir / 'lewm_epoch_*_object.ckpt')))
step_ckpts = sorted(glob.glob(str(run_dir / 'lewm_step_*_object.ckpt')))

latest = None
if step_ckpts:
    latest = step_ckpts[-1]
    print(f'Step checkpoints found: {len(step_ckpts)}')
    print(f'Latest step checkpoint: {latest}')
elif epoch_ckpts:
    latest = epoch_ckpts[-1]
    print(f'Epoch checkpoints found: {len(epoch_ckpts)}')
    print(f'Latest epoch checkpoint: {latest}')
else:
    canonical = str(run_dir / 'lewm_object.ckpt')
    if Path(canonical).exists():
        latest = canonical
        print(f'Checkpoint: {canonical} ({Path(canonical).stat().st_size / 1e6:.1f} MB)')
    else:
        raise FileNotFoundError('No checkpoint found under /content/stable-wm/pusht. Check /content/train_log.txt')

canonical = str(run_dir / 'lewm_object.ckpt')
if str(latest) != canonical:
    shutil.copy2(latest, canonical)
    print(f'Copied to: {canonical}')

CKPT_PATH = canonical
print(f'\nCheckpoint for eval: {CKPT_PATH}')


In [ ]:
# ── 8. Evaluate (CEM-MPC on PushT-v1, 100 episodes) ───────────────────────
# Uses CEM: 300 samples, 30 iterations, topk=30, horizon=5, 50 parallel envs
# Target: ~85% success rate (Fig 6 in paper)
# Runtime: ~20-30 min on GPU

import os
os.environ['MUJOCO_GL'] = 'egl'  # headless rendering
os.environ['STABLEWM_HOME'] = str(STABLEWM_HOME)

# Point stable-worldmodel to our data dir so it finds pusht_expert_train.h5
# (eval.py loads goal frames from the dataset)
data_link = STABLEWM_HOME / 'pusht_expert_train.h5'
assert data_link.exists(), f'Dataset not found: {data_link}'

os.chdir('/content/le-wm')
!python3 eval.py \
    --config-name=pusht.yaml \
    policy=pusht/lewm \
    eval.num_eval=100 \
    output.filename=pusht_results.txt \
    2>&1 | tee /content/eval_log.txt

print('Evaluation complete.')

In [ ]:
# ── 9. Print eval results ─────────────────────────────────────────────────
import glob

result_files = glob.glob(str(STABLEWM_HOME / '**' / 'pusht_results.txt'), recursive=True)
if not result_files:
    result_files = glob.glob('/content/le-wm/pusht_results.txt')

if result_files:
    with open(result_files[-1]) as f:
        print(f.read())
else:
    print('Results file not found. Tail of eval log:')
    !tail -30 /content/eval_log.txt

In [ ]:
# ── 10. Run diagnostics (T5 / T6 / sensitivity) ──────────────────────────
# Prefer local repo copy; fallback to raw GitHub download if missing.

import os, shutil

DIAG_URL = 'https://raw.githubusercontent.com/giuliovv/leworldduckie/main/src/pusht_diagnostics.py'
DIAG_PATH = '/content/pusht_diagnostics.py'
LOCAL_CANDIDATES = [
    '/content/leworldduckie/src/pusht_diagnostics.py',
    '/content/src/pusht_diagnostics.py',
]

local_src = next((p for p in LOCAL_CANDIDATES if os.path.exists(p)), None)
if local_src:
    shutil.copy(local_src, DIAG_PATH)
    print(f'Using local diagnostics script: {local_src}')
else:
    import urllib.request
    urllib.request.urlretrieve(DIAG_URL, DIAG_PATH)
    print('Downloaded diagnostics script from GitHub')

os.chdir('/content')
!python3 pusht_diagnostics.py     --ckpt {CKPT_PATH}     --data {H5_PATH}     --n-samples 200     --out /content/pusht_diagnostics.txt

In [ ]:
# ── 11. [Optional] Upload checkpoint + results to S3 ──────────────────────
# Uncomment and fill in credentials if you want to save to S3.
#
# import boto3
# from pathlib import Path
#
# S3_BUCKET = 'leworldduckie'
# S3_PREFIX = 'training/pusht'
# REGION    = 'us-east-1'
#
# s3 = boto3.client('s3', region_name=REGION)
# run_dir = STABLEWM_HOME / 'pusht'
#
# uploads = [
#     (CKPT_PATH,                             f'{S3_PREFIX}/pusht/lewm_object.ckpt'),
#     ('/content/train_log.txt',              f'{S3_PREFIX}/train_log.txt'),
#     ('/content/eval_log.txt',               f'{S3_PREFIX}/eval_log.txt'),
#     ('/content/pusht_diagnostics.txt',      f'{S3_PREFIX}/pusht_diagnostics.txt'),
# ]
#
# for src, key in uploads:
#     if Path(src).exists():
#         s3.upload_file(src, S3_BUCKET, key)
#         print(f'Uploaded {key}')
#
# # Also upload all epoch checkpoints
# import glob
# for ckpt in glob.glob(str(run_dir / 'lewm_epoch_*_object.ckpt')):
#     key = f"{S3_PREFIX}/pusht/{Path(ckpt).name}"
#     s3.upload_file(ckpt, S3_BUCKET, key)
#     print(f'Uploaded {key}')

print('S3 upload cell — uncomment to use')

## Expected outcomes

| Metric | Paper target | Pass threshold |
|--------|-------------|----------------|
| MPC success rate | ~85% | ≥75% (±10%) |
| T5 R² (block_x/y) | > 0.99 | > 0.95 |
| T6 ratio (action discriminability) | not reported | > 2× |
| Sensitivity ratio | not reported | < 10× (action-sensitive) |

## Interpreting diagnostics vs Duckietown

| | Duckietown ep20 | Push-T (expected) |
|---|---|---|
| T6 ratio | 0.86× FAIL | > 2× PASS |
| Sensitivity ratio | 50.6× (INPUT-DOMINATED) | < 10× (action matters) |
| Verdict | Encoder redundancy | Architecture works |

**If Push-T passes all three:** Pipeline is validated. Duckietown failure is task-specific — encoder can predict next frame without actions because lane changes are slow and predictable from camera geometry alone.

**If Push-T also fails T6 or sensitivity:** Pipeline bug, not task-specific. Check data normalization, frameskip config, or action dimension mismatch.